In [5]:
# ============================================================================
# REAL DATASET: Load Drugs + Proteins from Disk
# ============================================================================

import os
import time
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

# ============================================================================
# Dataset Class
# ============================================================================

class DTADataset(Dataset):
    """
    Drug-Target Affinity Dataset
    Loads pre-processed drugs and proteins from disk
    """
    
    def __init__(self, csv_file='data/Davis/Davis.csv', 
                 drugs_dir='drugs', 
                 proteins_dir='proteins',
                 embeddings_dir='data/Davis/processed/ESM_embedding_pocket'):
        """
        Args:
            csv_file: CSV with COMPOUND_ID, PROTEIN_ID, REG_LABEL
            drugs_dir: Directory with saved drug .pt files
            proteins_dir: Directory with saved protein .pt files (tensor dicts)
            embeddings_dir: Directory with ESM embedding .npy files
        """
        print(f"📂 Loading dataset from: {csv_file}")
        start = time.time()
        
        # Read CSV
        self.df = pd.read_csv(csv_file)
        self.drugs_dir = drugs_dir
        self.proteins_dir = proteins_dir
        self.embeddings_dir = embeddings_dir
        
        print(f"✓ Loaded {len(self.df)} drug-protein pairs in {time.time()-start:.2f}s")
        print(f"✓ Drugs directory: {drugs_dir}/")
        print(f"✓ Proteins directory: {proteins_dir}/")
        print(f"✓ Embeddings directory: {embeddings_dir}/")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        """
        Load one drug-protein pair
        """
        row = self.df.iloc[idx]
        
        compound_id = row['COMPOUND_ID']
        protein_id = row['PROTEIN_ID']
        label = float(row['REG_LABEL'])
        
        # Load drug graph (already PyG Data object)
        drug_path = os.path.join(self.drugs_dir, f'{compound_id}.pt')
        drug_graph = torch.load(drug_path)
        
        # Load protein tensor dict and convert to PyG Data
        protein_path = os.path.join(self.proteins_dir, f'{protein_id}.pt')
        protein_dict = torch.load(protein_path)
        
        # Wrap protein tensors in PyG Data object
        protein_graph = Data(
            x=protein_dict['x'],
            edge_index=protein_dict['edge_index'],
            edge_attr=protein_dict['edge_attr'],
            laplacian_pe=protein_dict['laplacian_pe']
        )
        
        # Load ESM embedding
        esm_path = os.path.join(self.embeddings_dir, f'{protein_id}.npy')
        protein_embedding = np.load(esm_path, allow_pickle=True)
        
        return drug_graph, protein_graph, protein_embedding, label


# ============================================================================
# Custom Collate Function
# ============================================================================

def custom_collate_dta(batch):
    """
    Collate function for batching drug-protein pairs
    Exactly replicates the dummy version
    """
    drug_graphs = [item[0] for item in batch]
    protein_graphs = [item[1] for item in batch]
    protein_embeddings = [item[2] for item in batch]
    labels = [item[3] for item in batch]
    
    # Batch graphs using PyG
    drug_batch = Batch.from_data_list(drug_graphs)
    protein_batch = Batch.from_data_list(protein_graphs)
    
    # Pad protein embeddings to max length
    max_protein_len = max(emb.shape[0] for emb in protein_embeddings)
    padded_embeddings = []
    
    for emb in protein_embeddings:
        if emb.shape[0] < max_protein_len:
            pad_size = max_protein_len - emb.shape[0]
            padded = np.pad(emb, ((0, pad_size), (0, 0)), mode='constant', constant_values=0)
            padded_embeddings.append(padded)
        else:
            padded_embeddings.append(emb)
    
    protein_embedding_batch = torch.FloatTensor(np.array(padded_embeddings))
    label_batch = torch.FloatTensor(labels)
    
    return drug_batch, protein_batch, protein_embedding_batch, label_batch


# ============================================================================
# Test the Dataset
# ============================================================================

print("="*70)
print("🚀 REAL DATASET LOADING TEST")
print("="*70)

# Create dataset
dataset_start = time.time()
dataset = DTADataset(
    csv_file='data/Davis/Davis.csv',
    drugs_dir='drugs',
    proteins_dir='proteins',
    embeddings_dir='data/Davis/processed/ESM_embedding_pocket'
)
dataset_time = time.time() - dataset_start

print(f"\n{'='*70}")
print(f"📊 Dataset Statistics")
print(f"{'='*70}")
print(f"Total samples: {len(dataset)}")
print(f"Dataset creation time: {dataset_time:.3f}s")

# Test single item loading
print(f"\n{'='*70}")
print(f"🔍 Testing Single Item Load")
print(f"{'='*70}")

item_start = time.time()
drug_graph, protein_graph, protein_emb, label = dataset[0]
item_time = time.time() - item_start

print(f"\nSample 0 (loaded in {item_time:.4f}s):")
print(f"  Drug Graph:")
print(f"    - Atoms: {drug_graph.x.shape[0]}")
print(f"    - Node features: {drug_graph.x.shape}")
print(f"    - Edges: {drug_graph.edge_index.shape[1]}")
print(f"    - Edge features: {drug_graph.edge_attr.shape}")
print(f"    - Laplacian PE: {drug_graph.laplacian_pe.shape}")
print(f"  Protein Graph:")
print(f"    - Residues: {protein_graph.x.shape[0]}")
print(f"    - Node features: {protein_graph.x.shape}")
print(f"    - Edges: {protein_graph.edge_index.shape[1]}")
print(f"    - Edge features: {protein_graph.edge_attr.shape}")
print(f"    - Laplacian PE: {protein_graph.laplacian_pe.shape}")
print(f"  ESM Embedding: {protein_emb.shape}")
print(f"  Label: {label:.4f}")

# Create DataLoader
print(f"\n{'='*70}")
print(f"🔄 Creating DataLoader (batch_size=4)")
print(f"{'='*70}")

dataloader = DataLoader(
    dataset, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=custom_collate_dta,
    num_workers=0
)

print(f"✓ DataLoader created with {len(dataloader)} batches")

# Test batching
print(f"\n{'='*70}")
print(f"📦 Testing Batch Loading")
print(f"{'='*70}")

batch_start = time.time()
for batch_idx, (drug_batch, protein_batch, protein_emb_batch, labels) in enumerate(dataloader):
    batch_time = time.time() - batch_start
    
    print(f"\n✓ Batch {batch_idx + 1} (loaded in {batch_time:.4f}s):")
    print(f"  Drug Batch:")
    print(f"    - Total atoms in batch: {drug_batch.x.shape[0]}")
    print(f"    - Node features shape: {drug_batch.x.shape}")
    print(f"    - Total edges: {drug_batch.edge_index.shape[1]}")
    print(f"    - Edge features shape: {drug_batch.edge_attr.shape}")
    print(f"    - Batch assignment: {drug_batch.batch.shape}")
    print(f"    - Num graphs in batch: {drug_batch.num_graphs}")
    print(f"    - Laplacian PE shape: {drug_batch.laplacian_pe.shape}")
    
    print(f"  Protein Batch:")
    print(f"    - Total residues in batch: {protein_batch.x.shape[0]}")
    print(f"    - Node features shape: {protein_batch.x.shape}")
    print(f"    - Total edges: {protein_batch.edge_index.shape[1]}")
    print(f"    - Edge features shape: {protein_batch.edge_attr.shape}")
    print(f"    - Batch assignment: {protein_batch.batch.shape}")
    print(f"    - Num graphs in batch: {protein_batch.num_graphs}")
    print(f"    - Laplacian PE shape: {protein_batch.laplacian_pe.shape}")
    
    print(f"  Protein Embeddings Batch: {protein_emb_batch.shape}")
    print(f"    - (batch_size={protein_emb_batch.shape[0]}, max_residues={protein_emb_batch.shape[1]}, features={protein_emb_batch.shape[2]})")
    
    print(f"  Labels: {labels.shape}")
    print(f"    - Values: {labels.tolist()}")
    
    break
    
    batch_start = time.time()

print(f"\n{'='*70}")
print(f"✅ SUCCESS! Dataset and DataLoader working perfectly!")
print(f"{'='*70}")
print(f"\n📋 Summary:")
print(f"  - Total samples: {len(dataset)}")
print(f"  - Drugs loaded from: {dataset.drugs_dir}/")
print(f"  - Proteins loaded from: {dataset.proteins_dir}/")
print(f"  - ESM embeddings from: {dataset.embeddings_dir}/")
print(f"  - Batch size: 4")
print(f"  - Ready for training! 🚀")
print(f"{'='*70}")


🚀 REAL DATASET LOADING TEST
📂 Loading dataset from: data/Davis/Davis.csv
✓ Loaded 24956 drug-protein pairs in 0.10s
✓ Drugs directory: drugs/
✓ Proteins directory: proteins/
✓ Embeddings directory: data/Davis/processed/ESM_embedding_pocket/

📊 Dataset Statistics
Total samples: 24956
Dataset creation time: 0.097s

🔍 Testing Single Item Load

Sample 0 (loaded in 0.0010s):
  Drug Graph:
    - Atoms: 27
    - Node features: torch.Size([27, 44])
    - Edges: 60
    - Edge features: torch.Size([60, 10])
    - Laplacian PE: torch.Size([27, 8])
  Protein Graph:
    - Residues: 13
    - Node features: torch.Size([13, 41])
    - Edges: 54
    - Edge features: torch.Size([54, 5])
    - Laplacian PE: torch.Size([13, 8])
  ESM Embedding: (13, 320)
  Label: 7.3665

🔄 Creating DataLoader (batch_size=4)
✓ DataLoader created with 6239 batches

📦 Testing Batch Loading

✓ Batch 1 (loaded in 0.0058s):
  Drug Batch:
    - Total atoms in batch: 137
    - Node features shape: torch.Size([137, 44])
    - Tota